In [166]:
# Bibliotecas para manipulação de dados
import pandas as pd
import numpy as np

# Bibliotecas para visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas para pré-processamento e divisão de dados
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Bibliotecas para os modelos de machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Bibliotecas para métricas de avaliação
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

In [167]:
train_df = pd.read_csv("C:/Users/user/Documents/EBAC/dados/train.csv", delimiter=',')
test_df = pd.read_csv("C:/Users/user/Documents/EBAC/dados/test.csv", delimiter=',')

In [168]:
# Preenchendo valores nulos em Age com a média
train_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
test_df['Age'].fillna(test_df['Age'].mean(), inplace=True)
test_df['Fare'].fillna(test_df['Fare'].mean(), inplace=True)

C:\Users\user\AppData\Local\Temp\ipykernel_1144\217807486.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(train_df['Age'].mean(), inplace=True)
C:\Users\user\AppData\Local\Temp\ipykernel_1144\217807486.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [169]:
# 1. Para a variável 'Sex', podemos usar Label Encoding, pois há apenas duas categorias
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})

In [170]:
# 2. Para a variável 'Embarked', podemos usar One-Hot Encoding, pois há mais de duas categorias
train_df = pd.get_dummies(train_df, columns=['Embarked'], prefix='Embarked')
test_df = pd.get_dummies(test_df, columns=['Embarked'], prefix='Embarked')

In [171]:
# 3. Para a variável 'Cabin', podemos apenas criar uma nova variável indicando se a cabine é conhecida ou não
train_df['Cabin_known'] = train_df['Cabin'].notna().astype(int)
test_df['Cabin_known'] = test_df['Cabin'].notna().astype(int)

# Agora podemos descartar a coluna 'Cabin' original
train_df = train_df.drop(columns=['Cabin'])
test_df_teste = test_df.drop(columns=['Cabin'])

In [172]:
train_df = train_df.drop(columns=['Name','Ticket'])

In [173]:
variaveis = ['Sex', 'Cabin_known','Fare','Pclass']

In [174]:
# Separando as variáveis dependentes (Y) e independentes (X) no conjunto de treino
X = train_df[variaveis]
y = train_df['Survived']

X_test = test_df[variaveis]

In [175]:
# Pouco pre processamento

In [176]:
# Criando o modelo de random forest
rf_duelo1 = RandomForestClassifier(random_state=42)

In [177]:
# Treinando o modelo da floresta com os dados de treino
rf_duelo1.fit(X, y)

RandomForestClassifier(random_state=42)

In [178]:
# Fazendo previsões com o modelo treinado nos dados de treino
Y_floresta_duelo1 = rf_duelo1.predict(X)

In [179]:
relatorio = classification_report(y, Y_floresta_duelo1)
print("Relatório de Classificação:")
print(relatorio)

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93       549
           1       0.91      0.86      0.88       342

    accuracy                           0.91       891
   macro avg       0.91      0.90      0.91       891
weighted avg       0.91      0.91      0.91       891



In [180]:
# Com pre processamento

In [181]:
# Balanceando os dados com SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, Y_train_balanced = smote.fit_resample(X, y)

In [182]:
# Padronizando os dados
scaler = StandardScaler()
# Ajustando e transformando os dados de treino balanceados
X_train_balanced_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

In [183]:
print(f"Tamanho do conjunto de treino balanceado: {X_train_balanced_scaled.shape[0]}")
print(f"Tamanho do conjunto de teste: {X_test_scaled.shape[0]}")

Tamanho do conjunto de treino balanceado: 1098
Tamanho do conjunto de teste: 418


In [184]:
print("Distribuição das classes antes do balanceamento:")
print(y.value_counts())

Distribuição das classes antes do balanceamento:
Survived
0    549
1    342
Name: count, dtype: int64


In [185]:
print("\nDistribuição das classes depois do balanceamento:")
print(Y_train_balanced.value_counts())


Distribuição das classes depois do balanceamento:
Survived
0    549
1    549
Name: count, dtype: int64


In [186]:
floresta_duelo2 = RandomForestClassifier(random_state=42)

In [187]:
floresta_duelo2.fit(X_train_balanced_scaled, Y_train_balanced)

RandomForestClassifier(random_state=42)

In [188]:
Y_floresta_duelo2 = floresta_duelo2.predict(X_train_balanced_scaled)

In [189]:
relatorio = classification_report(Y_train_balanced, Y_floresta_duelo2)
print("Relatório de Classificação:")
print(relatorio)

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.90      0.93      0.92       549
           1       0.93      0.90      0.92       549

    accuracy                           0.92      1098
   macro avg       0.92      0.92      0.92      1098
weighted avg       0.92      0.92      0.92      1098



In [190]:
# Melhorando hyperparametros

In [191]:
floresta_duelo3 = RandomForestClassifier(random_state=42)

In [192]:
param_grid = {
    'n_estimators': [20, 50, 100, 200],  # Número de árvores na floresta
    'max_depth': [None, 5, 10, 20, 30],  # Profundidade máxima de cada árvore
    'min_samples_split': [2, 5, 10],     # Número mínimo de amostras necessárias para dividir um nó
    'min_samples_leaf': [1, 2, 4, 6],     # Número mínimo de amostras necessárias para estar em um nó folha
    'max_features': ['sqrt', 'log2', None], # Número máximo de recursos considerados para dividir um nó
    'bootstrap': [True, False],         # Se amostras com reposição são usadas ao construir árvores
    'oob_score': [True, False],         # Se usar amostras fora da bolsa (out-of-bag) para estimar o erro de generalização
    'criterion': ['gini', 'entropy'],     # A função para medir a qualidade de uma divisão
    'min_impurity_decrease': [0.0, 0.001, 0.005] # Um nó será dividido se essa divisão induzir uma diminuição da impureza maior ou igual a esse valor
}

In [193]:
grid_search_floresta = GridSearchCV(floresta_duelo3, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

In [ ]:
# Executando o Randomized Search
grid_search_floresta.fit(X_train_balanced_scaled, Y_train_balanced)

In [ ]:
best_params_floresta = grid_search_floresta.best_params_
print(f"Melhores parâmetros: {best_params_floresta}")

In [ ]:
best_floresta_model = grid_search_floresta.best_estimator_
best_floresta_model.fit(X_train_balanced_scaled, Y_train_balanced)

In [ ]:
Y_pred_arvore = best_floresta_model.predict(X_train_balanced_scaled)
report_arvore = classification_report(Y_train_balanced, Y_pred_arvore)
print("Relatório de Métricas - Modelo com Grid Search (Árvore de Decisão):\n", report_arvore)

In [ ]:
Y_pred_forest_sub = floresta_duelo2.predict(X_test_scaled)

In [ ]:
Y_pred_forest_sub

In [ ]:
sub_forest = pd.Series(Y_pred_forest_sub, index=test_df['PassengerId'], name = 'Survived')

In [ ]:
sub_forest.to_csv("C:/Users/user/Documents/EBAC/dados/sub_forest.csv" , header=True)